In [ ]:
# ETL stands for

# Extract: extract the data from the different sources

# Transform: Transform the unstructured data into structured data. Transformations like cleaning, manipulation, etc.

# Load : Load the transformed data into a location or date warehouse.


In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, lit, floor, rand

# Create Spark session
spark = SparkSession.builder.appName("ETLPractice").getOrCreate()

# Corrected file paths using raw strings
source_path = "/content/loan.csv"
target_path = "loan_result.csv"

# Read the source CSV file
load_data = spark.read.csv(source_path, header=True, inferSchema=True)

# Show loaded data
load_data.show(5)

# Write the data back to a new CSV file
load_data.write.csv(target_path, header=True, mode='overwrite')















+-----------+---+------+------------+--------------+-----------+------+-----------+-------------+-------------+-----------+-------+------------+----------------+------------------+
|Customer_ID|Age|Gender|  Occupation|Marital Status|Family Size|Income|Expenditure|Use Frequency|Loan Category|Loan Amount|Overdue| Debt Record| Returned Cheque| Dishonour of Bill|
+-----------+---+------+------------+--------------+-----------+------+-----------+-------------+-------------+-----------+-------+------------+----------------+------------------+
|    IB14001| 30|  MALE|BANK MANAGER|        SINGLE|          4| 50000|      22199|            6|      HOUSING| 10,00,000 |      5|      42,898|               6|                 9|
|    IB14008| 44|  MALE|   PROFESSOR|       MARRIED|          6| 51000|      19999|            4|     SHOPPING|     50,000|      3|      33,999|               1|                 5|
|    IB14012| 30|FEMALE|     DENTIST|        SINGLE|          3| 58450|      27675|            

In [13]:
load_data.columns
load_data.show(5)


+-----------+---+------+------------+--------------+-----------+------+-----------+-------------+-------------+-----------+-------+------------+----------------+------------------+
|Customer_ID|Age|Gender|  Occupation|Marital Status|Family Size|Income|Expenditure|Use Frequency|Loan Category|Loan Amount|Overdue| Debt Record| Returned Cheque| Dishonour of Bill|
+-----------+---+------+------------+--------------+-----------+------+-----------+-------------+-------------+-----------+-------+------------+----------------+------------------+
|    IB14001| 30|  MALE|BANK MANAGER|        SINGLE|          4| 50000|      22199|            6|      HOUSING| 10,00,000 |      5|      42,898|               6|                 9|
|    IB14008| 44|  MALE|   PROFESSOR|       MARRIED|          6| 51000|      19999|            4|     SHOPPING|     50,000|      3|      33,999|               1|                 5|
|    IB14012| 30|FEMALE|     DENTIST|        SINGLE|          3| 58450|      27675|            

In [16]:
print(load_data.columns)


['Customer_ID', 'Age', 'Gender', 'Occupation', 'Marital Status', 'Family Size', 'Income', 'Expenditure', 'Use Frequency', 'Loan Category', 'Loan Amount', 'Overdue', ' Debt Record', ' Returned Cheque', ' Dishonour of Bill']


In [18]:
# Transformation 1: Concatenate Customer + Occupation + MaritalSta into 'customer_info'
load_data = load_data.withColumn(
              "customer_info",
               concat_ws(" | ", col("Customer_ID"), col("Occupation"), col("Marital Status")))

load_data.select("Customer_ID", "customer_info").show(5, truncate=False)



+-----------+-------------------------------+
|Customer_ID|customer_info                  |
+-----------+-------------------------------+
|IB14001    |IB14001 | BANK MANAGER | SINGLE|
|IB14008    |IB14008 | PROFESSOR | MARRIED  |
|IB14012    |IB14012 | DENTIST | SINGLE     |
|IB14018    |IB14018 | TEACHER | MARRIED    |
|IB14022    |IB14022 | POLICE | SINGLE      |
+-----------+-------------------------------+
only showing top 5 rows



In [19]:
# Transformation 2: calculate net salary as Income minus 10% tax
load_data = load_data.withColumn(
             "net_salary",
             floor(col("Income") - (col("Income") * 0.10)))

# Show first 10 rows
load_data.select("Customer_ID", "Income", "net_salary").show(10, truncate=False)


+-----------+------+----------+
|Customer_ID|Income|net_salary|
+-----------+------+----------+
|IB14001    |50000 |45000     |
|IB14008    |51000 |45900     |
|IB14012    |58450 |52605     |
|IB14018    |45767 |41190     |
|IB14022    |43521 |39168     |
|IB14024    |34999 |31499     |
|IB14025    |46619 |41957     |
|IB14027    |49999 |44999     |
|IB14029    |45008 |40507     |
|IB14031    |55999 |50399     |
+-----------+------+----------+
only showing top 10 rows



In [20]:
# Add random 'age' column between 20 and 50
load_data = load_data.withColumn('age', floor(lit(20) + rand() * lit(31)))

# Show results
load_data.select("Customer_ID", "age").show(10)

+-----------+---+
|Customer_ID|age|
+-----------+---+
|    IB14001| 30|
|    IB14008| 35|
|    IB14012| 31|
|    IB14018| 42|
|    IB14022| 38|
|    IB14024| 25|
|    IB14025| 41|
|    IB14027| 43|
|    IB14029| 26|
|    IB14031| 45|
+-----------+---+
only showing top 10 rows



In [21]:
# Filter customers with age >= 30
load_data = load_data.filter(col('age') >= 30)

# Show filtered result
load_data.show()

+-----------+---+------+-----------------+--------------+-----------+------+-----------+-------------+------------------+-----------+-------+------------+----------------+------------------+--------------------+----------+
|Customer_ID|age|Gender|       Occupation|Marital Status|Family Size|Income|Expenditure|Use Frequency|     Loan Category|Loan Amount|Overdue| Debt Record| Returned Cheque| Dishonour of Bill|       customer_info|net_salary|
+-----------+---+------+-----------------+--------------+-----------+------+-----------+-------------+------------------+-----------+-------+------------+----------------+------------------+--------------------+----------+
|    IB14001| 30|  MALE|     BANK MANAGER|        SINGLE|          4| 50000|      22199|            6|           HOUSING| 10,00,000 |      5|      42,898|               6|                 9|IB14001 | BANK MA...|     45000|
|    IB14008| 35|  MALE|        PROFESSOR|       MARRIED|          6| 51000|      19999|            4|      

In [22]:
# Transformation 4: Group by Age and Calculate Average Salary
avg_salary_by_age = load_data.groupBy('age').agg({'net_salary' :'avg'}).withColumnRenamed('avg(salary)', 'avg_salary')
avg_salary_by_age.show()

+---+------------------+
|age|   avg(net_salary)|
+---+------------------+
| 34| 47362.28571428572|
| 50| 39783.38461538462|
| 43| 96133.70588235294|
| 32| 67784.77777777778|
| 31| 86975.68421052632|
| 39|53197.333333333336|
| 41| 52635.57142857143|
| 33| 87684.61111111111|
| 48|         53990.375|
| 44|        56490.0625|
| 37|57190.666666666664|
| 49| 55218.36363636364|
| 35| 52563.13333333333|
| 36| 49993.78571428572|
| 38| 45310.61111111111|
| 30|          55584.75|
| 42|           40778.2|
| 46|42903.117647058825|
| 40| 48041.53846153846|
| 45| 52792.28571428572|
+---+------------------+
only showing top 20 rows



In [23]:
load_data = load_data.orderBy("age")
load_data.show()

+-----------+---+------+-----------------+--------------+-----------+------+-----------+-------------+------------------+-----------+-------+------------+----------------+------------------+--------------------+----------+
|Customer_ID|age|Gender|       Occupation|Marital Status|Family Size|Income|Expenditure|Use Frequency|     Loan Category|Loan Amount|Overdue| Debt Record| Returned Cheque| Dishonour of Bill|       customer_info|net_salary|
+-----------+---+------+-----------------+--------------+-----------+------+-----------+-------------+------------------+-----------+-------+------------+----------------+------------------+--------------------+----------+
|    IBI4155| 30|FEMALE|SOFTWARE ENGINEER|        SINGLE|          4| 55680|      29000|            5|        AUTOMOBILE|  7,89,000 |      5|      24,000|               6|                 4|IBI4155 | SOFTWAR...|     50112|
|    IBI4202| 30|  MALE|     BANK MANAGER|       MARRIED|          4| 67500|      25780|            6|      

In [24]:
# Save the transformed data to an external CSV file
load_data.write.csv(target_path, mode='overwrite', header=True)